# Language Detector — Character N-gram Classification

Language detection is one of the first steps in any multilingual NLP pipeline. Before you can translate, sentiment-analyse, or summarise a document, you need to know what language it is in. This project builds a language classifier that identifies 17 languages from character-level patterns — without relying on word dictionaries or external APIs.

Character n-grams are the right feature for this task: they capture script patterns (Cyrillic vs Latin vs Arabic), morphological patterns (German compound words, Spanish verb endings), and character combinations that are language-specific regardless of vocabulary.

This project is also personally relevant — I am bilingual in English and Spanish, and I wanted to understand what the model actually learns about the languages I speak.

In [ ]:
import json
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

from config import (
    ANALYZER, MAX_FEATURES, MODEL_DIR, NGRAM_RANGE,
    PLOTS_DIR, RANDOM_STATE, TEST_SIZE, TOP_N_LANGUAGES, TOP_N_NGRAMS
)
import explainer

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

MODEL_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

print(f'ANALYZER     : {ANALYZER}')
print(f'NGRAM_RANGE  : {NGRAM_RANGE}')
print(f'MAX_FEATURES : {MAX_FEATURES}')

## Why character n-grams for language detection

Word-level features require the model to have seen the exact words before. A word it has never seen is invisible. Character n-grams have no out-of-vocabulary problem — every text can be represented as a combination of character sequences, even if the words are new.

`analyzer='char_wb'` respects word boundaries, which is important: word-boundary patterns (how words start and end) are highly distinctive across languages. The German trigram `sch` and the Spanish bigram `qu` are reliable language signals regardless of which words contain them.

`ngram_range=(1, 3)` gives the model three levels of granularity:
- **Unigrams**: character-level script detection (Arabic characters vs Cyrillic vs Latin)
- **Bigrams**: letter-combination patterns distinctive to each language
- **Trigrams**: morphological fingerprints like German `sch`, Spanish `ció`, French `tion`

In [ ]:
# ── Data loading ─────────────────────────────────────────────────────────────

df = None

# Try kagglehub
try:
    import kagglehub
    path = kagglehub.dataset_download('basilb2s/language-detection')
    csv_files = list(Path(path).rglob('*.csv'))
    if csv_files:
        df = pd.read_csv(csv_files[0])
        print(f'Loaded from kagglehub: {csv_files[0]}')
except Exception as e:
    print(f'kagglehub unavailable: {e}')

# Try multiple GitHub mirrors in order
FALLBACK_URLS = [
    'https://raw.githubusercontent.com/amankharwal/Website-data/master/dataset.csv',
    'https://raw.githubusercontent.com/lyteabovenyte/NLP_Classification/main/Language_Detection.csv',
]
for url in FALLBACK_URLS:
    if df is not None:
        break
    try:
        df = pd.read_csv(url)
        print(f'Loaded from {url}: {len(df)} rows')
    except Exception as e:
        print(f'URL unavailable ({url}): {e}')

if df is None:
    raise RuntimeError(
        'All data sources failed. Install kagglehub and run: '
        'kagglehub.dataset_download("basilb2s/language-detection"), '
        'or place Language_Detection.csv in this directory and load it manually.'
    )

# Normalise columns
df.columns = df.columns.str.strip()
if 'language' in df.columns and 'Language' not in df.columns:
    df = df.rename(columns={'language': 'Language'})
if 'text' in df.columns and 'Text' not in df.columns:
    df = df.rename(columns={'text': 'Text'})

df = df.dropna(subset=['Text', 'Language'])
df['Text'] = df['Text'].str.strip()
df['Language'] = df['Language'].str.strip()
df = df[df['Text'].str.len() > 0].reset_index(drop=True)

print(f'\nShape: {df.shape}')
print(f'Languages ({df["Language"].nunique()}):', sorted(df['Language'].unique()))
print('\nClass distribution:')
print(df['Language'].value_counts().to_string())

In [ ]:
# ── Chart 01: Language distribution ──────────────────────────────────────────

dist = df['Language'].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(dist)))
dist.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Sample Count by Language', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Samples')
plt.tight_layout()
fig.savefig(PLOTS_DIR / '01_language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Chart 02: Text length by language ────────────────────────────────────────

df_plot = df.copy()
df_plot['text_length'] = df_plot['Text'].str.len()
order = df_plot.groupby('Language')['text_length'].median().sort_values().index.tolist()

fig2, ax2 = plt.subplots(figsize=(12, 7))
sns.boxplot(data=df_plot, x='text_length', y='Language', order=order,
            showfliers=False, ax=ax2, palette='viridis')
ax2.set_title('Text Length Distribution by Language', fontsize=14, fontweight='bold')
ax2.set_xlabel('Character Count')
ax2.set_ylabel('Language')
plt.tight_layout()
fig2.savefig(PLOTS_DIR / '02_text_length_by_language.png', dpi=150, bbox_inches='tight')
plt.show()

# ── One sample text per language ─────────────────────────────────────────────

print('\nOne sample text per language:')
for lang in sorted(df['Language'].unique()):
    sample = df[df['Language'] == lang]['Text'].iloc[0]
    print(f'\n[{lang}]')
    print(f'  {sample[:120].replace(chr(10), " ")}...')

## Model comparison — why three classifiers

**LinearSVC** is the standard for high-dimensional sparse text features — the same choice as in the spam-classifier project. It trains fast and typically achieves top accuracy on text classification. The implementation wraps it in `CalibratedClassifierCV` to get `predict_proba` for the UI confidence display.

**LogisticRegression** provides calibrated probabilities (`predict_proba`) directly, which are needed for the confidence scores in the UI. The multinomial formulation with the lbfgs solver handles 17-class classification cleanly.

**ComplementNB** was shown to outperform MultinomialNB on imbalanced text classification in the spam-classifier project — tested here for consistency. The three models cover discriminative linear (SVC), probabilistic linear (LR), and probabilistic generative (NB) approaches to the same problem.

In [ ]:
# ── Vectorise and split ───────────────────────────────────────────────────────

vectorizer = TfidfVectorizer(
    analyzer=ANALYZER,
    ngram_range=NGRAM_RANGE,
    max_features=MAX_FEATURES,
    sublinear_tf=True,
)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Language'])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['Text'], y,
    test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

# ── Train ─────────────────────────────────────────────────────────────────────

print('\nTraining LinearSVC...')
svc = LinearSVC(random_state=RANDOM_STATE, max_iter=2000)
svc.fit(X_train, y_train)
svc_cal = CalibratedClassifierCV(svc, cv='prefit')
svc_cal.fit(X_train, y_train)

print('Training LogisticRegression...')
lr = LogisticRegression(
    random_state=RANDOM_STATE, max_iter=1000,
    multi_class='multinomial', solver='lbfgs', n_jobs=-1
)
lr.fit(X_train, y_train)

print('Training ComplementNB...')
nb = ComplementNB()
nb.fit(X_train, y_train)

models = {'LinearSVC': svc_cal, 'LogisticRegression': lr, 'ComplementNB': nb}

# ── Evaluate ──────────────────────────────────────────────────────────────────

results = {}
print('\nModel comparison:')
print(f'{"Model":20s}  {"Accuracy":>10s}  {"Macro F1":>10s}')
print('-' * 45)
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    results[name] = {'accuracy': acc, 'macro_f1': f1, 'model': model, 'y_pred': y_pred}
    print(f'{name:20s}  {acc:10.4f}  {f1:10.4f}')

best_name = max(results, key=lambda k: results[k]['macro_f1'])
best_model = results[best_name]['model']
best_y_pred = results[best_name]['y_pred']
print(f'\nBest model: {best_name}')

In [ ]:
# ── Chart 05: Confusion matrix ────────────────────────────────────────────────

labels = label_encoder.classes_
cm = confusion_matrix(y_test, best_y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax,
            linewidths=0.5, vmin=0, vmax=1)
ax.set_title('Confusion Matrix (normalised) — Best Model', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(PLOTS_DIR / '05_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Chart 06: Model comparison ────────────────────────────────────────────────

names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
f1s = [results[n]['macro_f1'] for n in names]
x = np.arange(len(names))
width = 0.35

fig2, ax2 = plt.subplots(figsize=(9, 5))
bars1 = ax2.bar(x - width/2, accs, width, label='Accuracy', color='#7C3AED', alpha=0.85)
bars2 = ax2.bar(x + width/2, f1s, width, label='Macro F1', color='#A78BFA', alpha=0.85)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel('Score')
ax2.set_title('Model Comparison', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(names)
ax2.legend()
ax2.bar_label(bars1, fmt='%.4f', padding=3, fontsize=9)
ax2.bar_label(bars2, fmt='%.4f', padding=3, fontsize=9)
plt.tight_layout()
fig2.savefig(PLOTS_DIR / '06_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Chart 07: Per-language F1 ─────────────────────────────────────────────────

report = classification_report(y_test, best_y_pred,
                                target_names=labels, output_dict=True)
lang_f1 = {l: report[l]['f1-score'] for l in labels}
order = np.argsort(list(lang_f1.values()))
sorted_langs = [labels[i] for i in order]
sorted_f1s = [lang_f1[l] for l in sorted_langs]

fig3, ax3 = plt.subplots(figsize=(10, 7))
colors = ['#EF4444' if f < 0.9 else '#7C3AED' for f in sorted_f1s]
ax3.barh(sorted_langs, sorted_f1s, color=colors)
ax3.set_xlim(0, 1.05)
ax3.set_xlabel('F1 Score')
ax3.set_title('F1 Score per Language — Best Model', fontsize=13, fontweight='bold')
ax3.axvline(x=0.9, color='gray', linestyle='--', linewidth=0.8, label='0.90 threshold')
ax3.legend()
plt.tight_layout()
fig3.savefig(PLOTS_DIR / '07_per_language_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPer-language F1 (best model):')
for l in sorted(lang_f1, key=lang_f1.get, reverse=True):
    print(f'  {l:15s}: {lang_f1[l]:.4f}')

## N-gram analysis — what the model actually learned

The most interesting output of this project is not the accuracy — 99% on language detection is expected with character n-grams. The interesting output is the n-gram weight analysis: what character sequences did the model learn are most characteristic of each language?

This reveals genuine linguistic structure. The model learns that Arabic text contains patterns no Latin script language has. It learns that Russian Cyrillic differs from Greek in specific ways. It learns the morphological fingerprints of each language from character statistics alone.

The plot below shows the top 10 most distinctive character n-grams for six languages. These are the n-grams that, when present in a text, provide the strongest evidence for that language — the signals that the model has learned are language-specific.

In [ ]:
# ── Chart 04: Top n-grams per language ───────────────────────────────────────

def get_base_model(model):
    if hasattr(model, 'calibrated_classifiers_'):
        return model.calibrated_classifiers_[0].estimator
    return model

base = get_base_model(best_model)

if hasattr(base, 'coef_'):
    feature_names = np.array(vectorizer.get_feature_names_out())
    classes = label_encoder.classes_
    featured = ['English', 'Spanish', 'French', 'German', 'Arabic', 'Russian']
    featured = [l for l in featured if l in classes]
    indices = [np.where(classes == l)[0][0] for l in featured]

    top_n = 10
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    for ax, lang, idx in zip(axes, featured, indices):
        top_idx = np.argsort(base.coef_[idx])[-top_n:][::-1]
        top_ngrams = feature_names[top_idx]
        top_weights = base.coef_[idx][top_idx]
        colors = plt.cm.viridis(np.linspace(0.3, 0.9, top_n))
        ax.barh(top_ngrams[::-1], top_weights[::-1], color=colors[::-1])
        ax.set_title(lang, fontsize=12, fontweight='bold')
        ax.set_xlabel('Coefficient weight')

    plt.suptitle('Top 10 Most Distinctive Character N-grams per Language',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / '04_top_ngrams_per_language.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Best model has no coef_ attribute — skipping n-gram plot')

## The explanation layer

`explainer.py` follows the same translation-layer pattern used in four previous projects: `shap_to_language.py` (telco churn), `text_explainer.py` (spam classifier), `risk_explainer.py` (credit risk), and the n-gram explanation here. The pattern is consistent: identify the features driving the model's decision, translate them into plain English, return them alongside the prediction.

This is not just a portfolio pattern — it is the design principle that makes ML outputs usable by non-technical people. The model's internal weight matrix contains the information needed to explain any prediction; the question is whether that information is surfaced or buried.

For language detection, the explanation layer:
1. Vectorises the input text with the same TF-IDF vectorizer used during training
2. Identifies which n-gram features are present in the input text
3. Looks up the model's coefficient weights for those features in the predicted language's row
4. Returns the top-weighted n-grams as evidence, with plain-English interpretation
5. Adds a script-type label and a one-sentence language description

In [ ]:
# ── Explanation layer demo ────────────────────────────────────────────────────

sample_texts = [
    ('English',  'The development of artificial intelligence has transformed how we interact with technology.'),
    ('Spanish',  'El desarrollo de la inteligencia artificial ha transformado la manera en que interactuamos.'),
    ('French',   'Le développement de l\'intelligence artificielle a transformé notre interaction avec la technologie.'),
    ('German',   'Die Entwicklung der künstlichen Intelligenz hat unsere Interaktion mit der Technologie verändert.'),
    ('Arabic',   'أدى تطور الذكاء الاصطناعي إلى تحويل طريقة تفاعلنا مع التكنولوجيا.'),
    ('Russian',  'Развитие искусственного интеллекта изменило способ взаимодействия человека с технологиями.'),
]

for true_lang, text in sample_texts:
    proba = best_model.predict_proba(vectorizer.transform([text]))[0]
    classes = label_encoder.classes_
    top_idx = np.argsort(proba)[-TOP_N_LANGUAGES:][::-1]
    top_languages = [
        {'rank': i+1, 'language': classes[idx], 'confidence': float(proba[idx])}
        for i, idx in enumerate(top_idx)
    ]
    result = explainer.explain_detection(
        text=text,
        top_languages=top_languages,
        vectorizer=vectorizer,
        model=best_model,
        label_encoder=label_encoder,
    )
    print(f'\n[{true_lang}]')
    print(f'  Detected  : {top_languages[0]["language"]} ({top_languages[0]["confidence"]*100:.1f}%)')
    print(f'  Script    : {result["script_type"]}')
    print(f'  Key n-grams: {", ".join(g["ngram"] for g in result["top_ngrams"])}')
    print(f'  Explanation: {result["explanation"]}')

## Personal context

Testing the model on Spanish — the language I grew up speaking — was the most interesting part of building this. The top n-grams the model identifies for Spanish include `que`, `es`, `ción`, `de`, `en`. Every Spanish speaker recognises these immediately as core patterns.

The model learned them from statistics, not from grammar rules. That is what makes character n-grams work: they encode linguistic structure implicitly, without being told what to look for. The pattern `ción` corresponds to the Spanish suffix for nouns derived from verbs (the equivalent of `-tion` in English, but with different spelling). The model did not need to know this — it learned it from the distribution of character sequences across 22,000 text samples.

For a bilingual person, this is a different kind of understanding than native intuition. I know `que` is common in Spanish because I use it constantly — *¿qué quieres?*, *lo que sea*, *porque*, *aunque*. The model knows it is common in Spanish because it appears at high frequency in Spanish text and at lower frequency in French and Portuguese. Same fact, different route to knowing it.

In [ ]:
# ── Save model artefacts ──────────────────────────────────────────────────────

import pickle, json
from sklearn.metrics import classification_report as cr

pickle.dump(best_model,     open(MODEL_DIR / 'best_model.pkl',   'wb'))
pickle.dump(vectorizer,     open(MODEL_DIR / 'vectorizer.pkl',   'wb'))
pickle.dump(label_encoder,  open(MODEL_DIR / 'label_encoder.pkl','wb'))
(MODEL_DIR / 'best_model_name.txt').write_text(best_name)

languages = sorted(df['Language'].unique().tolist())
(MODEL_DIR / 'language_list.json').write_text(json.dumps(languages, indent=2))

report = cr(y_test, best_y_pred, target_names=label_encoder.classes_, output_dict=True)
metrics = {
    'model_name': best_name,
    'accuracy': round(results[best_name]['accuracy'], 4),
    'macro_f1': round(results[best_name]['macro_f1'], 4),
    'per_language': {
        lang: {
            'precision': round(report[lang]['precision'], 4),
            'recall':    round(report[lang]['recall'],    4),
            'f1':        round(report[lang]['f1-score'],  4),
        }
        for lang in label_encoder.classes_
    },
    'all_models': {
        name: {
            'accuracy': round(r['accuracy'], 4),
            'macro_f1': round(r['macro_f1'], 4),
        }
        for name, r in results.items()
    }
}
(MODEL_DIR / 'model_metrics.json').write_text(json.dumps(metrics, indent=2))

print('Model artefacts saved:')
for f in sorted(MODEL_DIR.iterdir()):
    print(f'  {f.name}')

## Azure App Service Deployment

The API is deployed to Azure App Service on the F1 (free) tier. The deployment bundles all model files — the API loads them at startup and does not train at runtime.

### Deployment command sequence

```bash
# Create resource group and plan
az group create --name language-detector-rg --location westeurope
az appservice plan create --name language-detector-plan \
  --resource-group language-detector-rg --sku B1 --is-linux
# Scale to F1 via portal after creation

# Create web app
az webapp create --name language-detector-xoc \
  --resource-group language-detector-rg \
  --plan language-detector-plan --runtime "PYTHON:3.11"

# Configure startup
az webapp config set --name language-detector-xoc \
  --resource-group language-detector-rg \
  --startup-file "gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000 --timeout 600"

# Enable build during deployment
az webapp config appsettings set --name language-detector-xoc \
  --resource-group language-detector-rg \
  --settings SCM_DO_BUILD_DURING_DEPLOYMENT=true

# Zip and deploy
zip -r deploy.zip . -x "*.git*" -x "venv/*" -x "__pycache__/*" -x "*.ipynb_checkpoints*"
az webapp deployment source config-zip \
  --name language-detector-xoc \
  --resource-group language-detector-rg \
  --src deploy.zip
```

### Operational notes

- **Workers**: 1 worker with a 600s timeout. The model loads once at startup (~2s); inference is fast (<100ms per request).
- **Memory**: The TF-IDF matrix with 50,000 features and a linear classifier fits comfortably within F1 tier memory limits.
- **Cold start**: F1 tier apps sleep after 20 minutes of inactivity. The first request after sleep takes ~30s to wake the app and load the model.
- **Model updates**: Re-zip and re-deploy. No database migration needed — the API reads from `.pkl` files bundled in the deployment package.

## Key Findings

*TBD — fill in after running this notebook to completion.*

Populate with:
- Overall accuracy and macro F1 for the best model
- Winner model and comparison to the other two
- Easiest language to detect (expected: Arabic, Russian, Malayalam — non-Latin scripts)
- Hardest language to detect (expected: closely related languages like Spanish/Portuguese or Danish/Swedish)
- Top 3 most distinctive n-grams for English, Spanish, and one non-Latin script language
- Any surprising confusions in the confusion matrix